# Save and Load

https://www.tensorflow.org/tutorials/keras/save_and_load

## Import Keras

In [ ]:
import os

import tensorflow as tf
from tensorflow import keras

print(tf.version.VERSION)

## Dataset Example

In [ ]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

train_labels = train_labels[:1000]
test_labels = test_labels[:1000]

train_images = train_images[:1000].reshape(-1, 28 * 28) / 255.0
test_images = test_images[:1000].reshape(-1, 28 * 28) / 255.0

In [ ]:
def create_model():
    model = tf.keras.models.Sequential([
        keras.layers.Dense(512, activation='relu', input_shape=(784,)),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10)
    ])
    
    model.compile(
        optimizer='adam',
        loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[tf.metrics.SparseCategoricalAccuracy()]
    )
    
    return model

In [ ]:
model = create_model()

model.summary()

## Save checkpoints during training

In [ ]:
checkpoint_path = 'training_1/cp.ckpt'
checkpoint_dir = os.path.dirname(checkpoint_path)
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    verbose=1
)

model.fit(
    train_images,
    train_labels,
    epochs=10,
    validation_data=(test_images, test_labels),
    callbacks=[cp_callback]
)

In [ ]:
ls {checkpoint_dir}

In [ ]:
model = create_model()

lss, acc = model.evaluate(test_images, test_labels, verbose=2)

In [ ]:
model.load_weights(checkpoint_path)

loss, acc = model.evaluate(test_images, test_labels, verbose=2)

print("After restored, the model accuracy: {:5.2f}%".format(100 * acc))

## Checkpoint callback options

In [ ]:
checkpoint_path = 'training_2/cp-{epoch:04d}.ckpt'
checkpoint_dir = os.path.dirname(checkpoint_path)

batch_size = 32

cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    verbose=1,
    save_weights_only=True,
    save_freq=5*batch_size
)

model = create_model()

model.save_weights(checkpoint_path.format(epoch=0))

model.fit(
    train_images,
    train_labels,
    epochs=50,
    callbacks=[cp_callback],
    validation_data=(test_images, test_labels),
    verbose=0
)

In [ ]:
ls {checkpoint_dir}

In [ ]:
latest = tf.train.latest_checkpoint(checkpoint_dir)
latest

In [ ]:
model = create_model()

model.load_weights(latest)

loss, acc = model.evaluate(test_images, test_labels, verbose=2)
print("After restored, the model accuracy: {acc:5.2f}%".format(acc=100 * acc))

## Save the entire model

In [ ]:
model = create_model()

model.fit(train_images, train_labels, epochs=5)

In [ ]:
!mkdir -p saved_model
model.save('saved_model/my_model')

In [ ]:
ls saved_model

In [ ]:
ls saved_model\my_model

In [ ]:
new_model = tf.keras.models.load_model('saved_model/my_model')
new_model.summary()

### HDF5 format

In [ ]:
model = create_model()
model.fit(train_images, train_labels, epochs=5)

model.save('my_model.h5')

In [ ]:
new_model = tf.keras.models.load_model('my_model.h5')

new_model.summary()

In [ ]:
loss, acc = new_model.evaluate(test_images, test_labels, verbose=2)
print('Restored model, accuracy: {:5.2f}%'.format(100 * acc))